by Suman Kumar Bera
email id: skbera.iitkgp21@gmail.com<br>
LinkdIn: https://www.linkedin.com/in/skbera4/

### 🤖 Hands-On Tutorial: LLM Prompting and Structured Outputs

**What you'll learn today:**

| Part | Topic |
|---|---|
| 1 | Inference approaches (HuggingFace, HuggingFace API, OpenAI Client) |
| 2 | Prompting (summarization, QA, classification) |
| 3 | Tool Calling(Recap)
| 4 | Constrained generation & structured outputs (Pydantic, LlamaIndex, and Instructor) |

If you are using

In [ ]:
!pip install -q transformers accelerate huggingface_hub requests regex pydantic instructor openai sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.2/353.2 kB 8.6 MB/s eta 0:00:00


In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
print("Setup complete ✅")

GPU available: False
Setup complete ✅


---
# PART 1 — HuggingFace Inference Approaches + Prompting

## 1.1 Three ways to run an open-source LLM

| Approach | Needs internet? | Needs GPU locally? | Best for |
|---|---|---|---|
| **A. Transformers (local)** | ❌ No | Optional (faster with one) | Full control, fine-tuning |
| **B. HF Inference API (hosted)** | ✅ Yes | ❌ No | Quick prototyping, no setup |
| **C. vLLM (zero-config)** | ✅ Yes | ❌ No | Easiest big-model runs |

**Approach A (Transformers locally on the local CPU/GPU)**.

In [ ]:
# Load a small instruction-tuned model (Approach A: Transformers locally)
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # open, no gated-access login required
# Alternative (needs HF login + license accept): "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
)

print(f"Loaded {MODEL_ID} on", model.device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct on cpu


In [ ]:
import os

print(os.path.expanduser("~/.cache/huggingface"))

/root/.cache/huggingface


In [ ]:
# A small helper so we don't repeat boilerplate for the rest of the notebook

def ask(prompt, max_new_tokens=150, system=None, temperature=None):
    """Send a chat-style prompt to our local model and return the text reply."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    gen_kwargs = dict(max_new_tokens=max_new_tokens, do_sample=temperature is not None)
    if temperature:
        gen_kwargs["temperature"] = temperature

    outputs = model.generate(**inputs, **gen_kwargs)
    reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return reply.strip()

print("Helper function `ask()` ready!")

Helper function `ask()` ready!


In [ ]:
system = "Explain Simply"
prompt = "What is quantum computing in one sentence?"
messages = []
if system:
    messages.append({"role": "system", "content": system})
messages.append({"role": "user", "content": prompt})
print(messages)

[{'role': 'system', 'content': 'Explain Simply'}, {'role': 'user', 'content': 'What is quantum computing in one sentence?'}]


Q. What is the need for system and user roles?

In [ ]:
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
text

'<|im_start|>system\nExplain Simply<|im_end|>\n<|im_start|>user\nWhat is quantum computing in one sentence?<|im_end|>\n<|im_start|>assistant\n'

Q. What is the need for this type of chat template?

In [ ]:
inputs = tokenizer(text, return_tensors="pt").to(model.device)
inputs

{'input_ids': tensor([[151644,   8948,    198,    840,  20772,  28424, 151645,    198, 151644,
            872,    198,   3838,    374,  30128,  24231,    304,    825,  11652,
             30, 151645,    198, 151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
temperature = 0.3
gen_kwargs = dict(max_new_tokens=80, do_sample=temperature is not None)
gen_kwargs


{'max_new_tokens': 80, 'do_sample': True}

In [ ]:
outputs = model.generate(**inputs, **gen_kwargs)

In [ ]:
reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
reply

'Quantum computing uses the principles of quantum mechanics to perform calculations and process information, potentially solving complex problems much faster than classical computers.'

In [ ]:
# using local model downoaded from HuggingFace , Qwen

article = """
Antibiotics are a type of medication used to treat bacterial infections. They work by either
killing the bacteria or preventing them from reproducing, allowing the body's immune system to
fight off the infection. Antibiotics are usually taken orally in the form of pills, capsules, or
liquid solutions, or sometimes administered intravenously. They are not effective against viral
infections, and using them inappropriately can lead to antibiotic resistance.
"""

prompt = f"Summarize the following text in exactly one sentence:\n\n{article}"
summary = ask(prompt, max_new_tokens=80, temperature=0.3)
print("SUMMARY:\n", summary)

SUMMARY:
 Antibiotics are medications that kill bacteria or prevent their reproduction, treating bacterial infections but not viruses, emphasizing proper use to avoid resistance.


## 1.2 Approach B: HuggingFace hosted Inference API

No GPU or model download needed — just an HTTP request. Useful when you don't want to load a model into memory at all. (Requires a free token from huggingface.co/settings/tokens — **skip this cell if you don't have one**, it's optional context.)

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("hf_key")

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HF_TOKEN, #you can directly paste your token here
)

completion = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct:featherless-ai",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


## Approach 3: UpGrad hosted EndPoint

In [ ]:
from google.colab import userdata

API_KEY = userdata.get("api_key")

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://103.42.50.41:443/api1/",
    api_key=API_KEY, #you can directly paste your token here
)

DEFAULT_MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"

try:
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[
            {"role": "user", "content": "Reply with exactly: MODEL_WORKING"}
        ],
        max_tokens=10,
        temperature=0
    )

    print("✅ Model responded")
    print(response.choices[0].message.content)

except Exception as e:
    print("❌ Error")
    print(type(e).__name__)
    print(e)

NameError: name 'API_KEY' is not defined

In [ ]:
def ask_Qwen_7B(prompt, max_tokens=150, system=None, temperature=0):
    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
    )

    return response.choices[0].message.content.strip()

In [ ]:
print(ask_Qwen_7B("What is the capital of France?"))

The capital of France is Paris.


| Local transformers        | OpenAI API                          |
|----------------------------|--------------------------------------|
| `apply_chat_template()`    | server does it                      |
| `tokenizer()`               | server does it                      |
| `generate()`                | server does it                      |
| `decode()`                  | server does it                      |
| helper function optional   | helper function optional in chat    |

## 1.1 Prompting Task 1 — Classification

- **Zero-shot sentiment classification**


In [ ]:
classification_prompt = "Classify sentiment as exactly one word (positive/negative/neutral): 'The food was cold but the service was excellent.'"

print("temp=0.0:", [ask_Qwen_7B(classification_prompt, temperature=0.0, max_tokens=5) for _ in range(5)])
# print("temp=1.0:", [ask_Qwen_7B(classification_prompt, temperature=1.0, max_tokens=5) for _ in range(5)])
# print("temp=2.0:", [ask_Qwen_7B(classification_prompt, temperature=2.0, max_tokens=5) for _ in range(5)])
print("temp=3.0:", [ask_Qwen_7B(classification_prompt, temperature=3.0, max_tokens=5) for _ in range(5)])
# print("temp=4.0:", [ask_Qwen_7B(classification_prompt, temperature=4.0, max_tokens=5) for _ in range(5)])
print("temp=5.0:", [ask_Qwen_7B(classification_prompt, temperature=5.0, max_tokens=5) for _ in range(5)])


temp=0.0: ['Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral']
temp=3.0: ['Negative', 'neutral', 'Mixed', 'Neutral', '-neutral']
temp=5.0: ['-neutralekausta', 'Mixed', 'negative()\n:invoke again', '混合<tool_call>mouseup\n:variables', '-neutral']


In [ ]:
prompt = "Write one creative sentence describing a rainy city street."

temperatures = [0.0, 0.3, 0.7, 1.2, 1.9]

for temp in temperatures:
    print(f"\n=== temperature = {temp} ===")
    for run in range(3):  # run 3 times at each temp to show variance (or lack of it)
        print(f"  [{run+1}] {ask_Qwen_7B(prompt, temperature=temp)}")


=== temperature = 0.0 ===
  [1] The rain danced elegantly down the cobblestone streets, painting each puddle with shimmering reflections of neon signs and whispering secrets to the passersby in a language only the city and the downpour could understand.
  [2] The rain danced elegantly down the cobblestone streets, painting each puddle with shimmering reflections of neon signs and whispering secrets to the passersby in a language only the city and the downpour could understand.
  [3] The rain danced elegantly down the cobblestone streets, painting each puddle with shimmering reflections of neon signs and whispering secrets to the passersby in a language only the city and the downpour could understand.

=== temperature = 0.3 ===


RateLimitError: Error code: 429 - {'detail': 'Rate limit exceeded (20/min)'}

## 1.2 Prompting Task 2 — Text Summarization

Two common summarization styles:
- **Zero-shot instruction summarization** (give context + clear instruction)
- **TL;DR style** (ultra-short, audience-aware)

In [ ]:
summary_prompt = """
Summarize the following article in 2-3 sentences.

Article:
Large language models (LLMs) are deep learning models trained on massive amounts
of text data. They can perform tasks such as text generation, summarization,
translation, question answering, and code generation. Recent advances in
transformer architectures and scaling laws have significantly improved their
performance. However, LLMs can still generate incorrect information, known as
hallucinations, and require careful prompting and evaluation before being
deployed in critical applications.
"""

print(ask_Qwen_7B(summary_prompt, max_tokens=100, temperature=0.0))

In [ ]:
print("temperature=0.0")
print(ask_Qwen_7B(summary_prompt, max_tokens=100, temperature=0.0))

print("\ntemperature=0.8")
print(ask_Qwen_7B(summary_prompt, max_tokens=100, temperature=2.0))

In [ ]:
LEGAL_TEXT = """
A tenant must pay rent on or before the fifth day of each month.
If the rent is paid more than ten days after the due date without prior written approval from the landlord,
a late fee of $50 shall be charged.
"""

PROMPT = f"""
Read the following legal clause.

{LEGAL_TEXT}

Tasks:
1. Summarize the clause in at most two bullet points.
2. Generate a Python function

def calculate_late_fee(days_late):

that returns the appropriate late fee according to the clause.
"""

print(ask_Qwen_7B(PROMPT, max_tokens=200, temperature=0.0))

## 1.3 Prompting Task 3 — Question Answering

Two flavors:
- **Context-bound QA**: answer must come only from the given context (good for RAG / policy lookups)
- **Open-domain synthesis QA**: model uses its own knowledge to answer + format a list

In [ ]:
context = """
Our company policy states that employees get $500/year for professional development.
Reimbursement requests must be filed at least 30 days prior to the course start date.
"""
question = "Can I get reimbursed for a $200 course I bought yesterday without prior approval?"

prompt = f"""Answer the question using ONLY the context below. Keep it short. If unsure, say "Unsure about answer".

Context: {context}

Question: {question}

Answer:"""

print(ask_Qwen_7B(prompt, max_tokens=150, temperature=2.0))
print(ask(prompt, max_new_tokens=150))

In [ ]:
# Open-domain, formatted output
prompt = "List the three primary constraints of the James Webb Space Telescope's cooling system. Format as a numbered list."
print(ask(prompt, max_new_tokens=150))
print(ask_Qwen_7B(prompt, max_tokens=150))

### Reasoning

In [ ]:
# --- A reasoning problem that trips up zero-shot ---
problem = """A juggler has 16 balls. Half of the balls are golf balls,
and half of the golf balls are blue. How many blue golf balls are there? Give the answer directly."""

# ---------------------------------------------------------
# 1. ZERO-SHOT: just ask directly
# ---------------------------------------------------------
zero_shot_prompt = f"Question: {problem}\nAnswer:"

zero_shot_answer = ask(zero_shot_prompt, max_new_tokens=150, temperature=0.0)
print("Zero-shot answer:", zero_shot_answer)
# Smaller models often blurt "8" (they compute 16/2 and stop),
# instead of the correct 16/2/2 = 4.

In [ ]:
# --- A reasoning problem that trips up zero-shot ---
problem = """A juggler has 16 balls. Half of the balls are golf balls,
and half of the golf balls are blue. How many blue golf balls are there? Think step by step and then give the answer."""

# ---------------------------------------------------------
# 1. ZERO-SHOT: just ask directly
# ---------------------------------------------------------
zero_shot_prompt = f"Question: {problem}\nAnswer:"

zero_shot_answer = ask(zero_shot_prompt, max_new_tokens=150)
print("Zero-shot answer:", zero_shot_answer)
# Smaller models often blurt "8" (they compute 16/2 and stop),
# instead of the correct 16/2/2 = 4.

In [ ]:
zero_shot_prompt = """
Determine whether the sequence should be labeled ALPHA or BETA.

Sequence:
2, 5, 8, 11

Answer:
"""

print(ask_Qwen_7B(zero_shot_prompt, temperature=0.0, max_tokens=220))

In [ ]:
few_shot_prompt = """
Determine whether the sequence should be labeled ALPHA or BETA.

Example 1
Sequence:
1, 4, 7, 10
Answer: ALPHA

Example 2
Sequence:
3, 6, 9, 12
Answer: ALPHA

Example 3
Sequence:
2, 4, 8, 16
Answer: BETA

Example 4
Sequence:
5, 10, 20, 40
Answer: BETA

Now classify:

Sequence:
2, 5, 8, 11

Answer:
"""

print(ask_Qwen_7B(few_shot_prompt, temperature=0.0, max_tokens=120))

| Task                         | Recommended Temperature |
|-------------------------------|--------------------------|
| Classification                | 0                        |
| Extraction                    | 0                        |
| Summarization                 | 0–0.3                    |
| Coding                        | 0–0.2                    |
| Reasoning (self-consistency)  | 0.5–0.8                  |
| Brainstorming                 | 0.8–1.2                  |
| Creative writing              | 1.0+                     |

---
# PART 2.1 — Tool Calling
# PART 2.2 — Constrained Generation & Structured Outputs

LLMs generate text **token by token**, picking from a probability distribution (logits → softmax) over the whole vocabulary at every step. **Constrained generation** restricts which tokens are even allowed at each step, so the output is *guaranteed* to follow a structure — instead of hoping the model "behaves."

We'll go from the lowest level (manually masking logits) up to the easiest practical tool (Pydantic + Instructor).

In [ ]:
from IPython.display import Image, display

file_id = "1npxT0o8pKoHvYq4UpOk5BSeXTHxvWomG"

display(Image(
    url=f"https://drive.google.com/thumbnail?id={file_id}&sz=w1000"
))

In [ ]:
import numpy as np

# Tool implementation
def compute_determinant(matrix):
    matrix = np.array(matrix)
    det = np.linalg.det(matrix)
    return str(det)


tools = [
    {
        "type": "function",
        "function": {
            "name": "compute_determinant",
            "description": "Compute the determinant of a square matrix.",
            "parameters": {
                "type": "object",
                "properties": {
                    "matrix": {
                        "type": "array",
                        "description": "A square matrix.",
                        "items": {
                            "type": "array",
                            "items": {
                                "type": "number"
                            }
                        }
                    }
                },
                "required": ["matrix"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the internet for recent information.",
            # ...
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send an email to a recipient.",
            # ...
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_flight",
            "description": "Book a flight between two cities.",
            # ...
        }
    }
]

In [ ]:
# Generate a fixed random matrix
np.random.seed(42)
matrix = np.random.randint(-10, 11, size=(10, 10)).tolist()

PROMPT = f"""
Compute the determinant of the following matrix.

{matrix}
"""

# WITHOUT TOOL
response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[
        {
            "role": "user",
            "content": PROMPT
        }
    ],
    temperature=0
)

print("=" * 100)
print("WITHOUT TOOL")
print("=" * 100)
print(response.choices[0].message.content)

In [ ]:
# WITH TOOL
messages = [
    {
        "role": "user",
        "content": PROMPT
    }
]

response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",
    temperature=0
)

tool_call = response.choices[0].message.tool_calls[0]

In [ ]:
tool_call

In [ ]:
import json

args = json.loads(tool_call.function.arguments)
args

In [ ]:
determinant = compute_determinant(args["matrix"])
determinant

In [ ]:
messages.append(response.choices[0].message)

messages.append(
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(determinant)
    }
)

final_response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=messages,
    tools=tools,
    temperature=0
)

print("\n\n" + "=" * 100)
print("WITH TOOL")
print("=" * 100)
print(final_response.choices[0].message.content)


# Ground Truth (Computed Directly)
true_determinant = compute_determinant(matrix)

print("\n\n" + "=" * 100)
print("GROUND TRUTH (NUMPY)")
print("=" * 100)
print(f"Determinant = {true_determinant}")

### PyDantic

In [ ]:
!pip install llama-index
!pip install llama-index-readers-file
!pip install -q llama-index-llms-openai-like
!pip install -q instructor

In [ ]:
from llama_index.readers.file import PDFReader
from pathlib import Path

In [ ]:
pdf_reader = PDFReader()
documents = pdf_reader.load_data(file=Path("/content/DOC-20260704-WA0001..pdf"))
text = documents[0].text
print(text)

In [ ]:
from google.colab import userdata

API_KEY = userdata.get("api_key")

## A. Import Libraries

In [ ]:
from pathlib import Path
from datetime import datetime
from typing import List
from pydantic import BaseModel, Field, field_validator
from dateutil import parser as date_parser

### B. Load the PDF

In [ ]:
# ── Load the PDF text ───────────────────────────────────────────────────────
pdf_reader = PDFReader()
documents = pdf_reader.load_data(file=Path("/content/DOC-20260704-WA0001..pdf"))
text = documents[0].text
print(text)

NameError: name 'PDFReader' is not defined

In [ ]:
content = text

prompt = f"this is the bill: {content} tell me if the bill has been paid or not?"
print(ask_Qwen_7B(prompt, temperature=0.0))

NameError: name 'text' is not defined

In [ ]:
from IPython.display import Image, display

file_id = "1ctG_c6HvFUL-Oo0Lf8Y3uN6cO_g6hiMn"

display(Image(
    url=f"https://drive.google.com/thumbnail?id={file_id}&sz=w1000"
))

### C. PyDantic Class Creation

the doc: https://drive.google.com/file/d/1ioKROv-ybbWmHPiamkQcwKQShr2NUWKh/view?usp=sharing

In [ ]:
# ── Schema ───────────────────────────────────────────────────────────────
# Using real `datetime` fields is fine for both approaches below, since neither
# relies on the server's native tool-calling JSON-schema parser (that's what
# broke earlier with $defs/$ref). But the model can still return dates in
# inconsistent formats ("January 25, 2016" vs "2016-01-25"), so we add a
# validator to normalize whatever string comes back before Pydantic parses it
# into a datetime — this is what prevents the ValueError you hit before.
class Invoice(BaseModel):
    """A representation of information from an invoice."""

    vendor: str
    invoice_date: datetime
    due_date: datetime
    invoice_number: str
    items: List[str]

    # before validating the field, call this function
    @field_validator("invoice_date", "due_date", mode="before") #tells Pydantic when to run your validation function.
    @classmethod #tells Python how to call that function
    def normalize_date(cls, v):
        if isinstance(v, str):
            return date_parser.parse(v)
        return v

In [ ]:
from datetime import datetime

# Get current date and time
now = datetime.now()
print(now)

### D. LlamaIndex

In [ ]:
# ── Option A: llama-index, using text-based structured prediction ──────────
from llama_index.llms.openai_like import OpenAILike
from llama_index.core.program import LLMTextCompletionProgram
from llama_index.core.output_parsers import PydanticOutputParser

llm = OpenAILike(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    api_base="http://103.42.50.41:443/api1/",
    api_key=API_KEY,
    is_chat_model=True,
)

output_parser = PydanticOutputParser(output_cls=Invoice)

program = LLMTextCompletionProgram.from_defaults(
    output_parser=output_parser,
    prompt_template_str=(
        "Extract the invoice information from the text below.\n"
        "{format_instructions}\n\n"
        "Text:\n{text}\n"
    ),
    llm=llm,
    verbose=True,
)

response = program(
    text=text,
    format_instructions=output_parser.get_format_string(),
)
print("llama-index result:")
print(response)

ModuleNotFoundError: No module named 'llama_index'

In [ ]:
print(response.model_dump_json(indent=2))

### E. Instructor

In [ ]:
# ── Option B: instructor, JSON mode ─────────────────────────────────────
import instructor
from openai import OpenAI

client = OpenAI(
    base_url="http://103.42.50.41:443/api1/",
    api_key=API_KEY,
)

instructor_client = instructor.from_openai(client, mode=instructor.Mode.JSON)

extracted_invoice: Invoice = instructor_client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    response_model=Invoice,
    messages=[
        {"role": "user", "content": f"Extract the invoice details from this text:\n\n{text}"}
    ],
)

print("\ninstructor result:")
print(extracted_invoice)

# No manual parsing needed anymore -- these are already real datetime objects,
# normalized by the field_validator regardless of what string format the
# model originally returned.
print("\nInvoice date:", extracted_invoice.invoice_date)
print("Due date:", extracted_invoice.due_date)
print(type(extracted_invoice.invoice_date))

ModuleNotFoundError: No module named 'instructor'

In [ ]:
extracted_invoice.invoice_number

### F. USE CASE(to get the pending or paid amount and type of service)

In [ ]:
from typing import Optional
from enum import Enum

class ServiceType(str, Enum):
    """Constrain to a fixed set of categories so the model can't invent new ones."""
    web_design = "web_design"
    consulting = "consulting"
    development = "development"
    hosting = "hosting"
    marketing = "marketing"
    other = "other"

class LineItem(BaseModel):
    item_name: str = Field(description="Just the name of the service/product, no price or description")
    quantity: float = Field(description="Quantity or hours")
    price: float = Field(description="Price for this line item")
    service_type: ServiceType = Field(description="The category of service this line item represents")

class Invoice(BaseModel):
    vendor: str
    invoice_date: datetime
    due_date: datetime
    invoice_number: str
    line_items: List[LineItem]
    total_amount: float = Field(description="The final total amount due on the invoice, including tax")
    is_paid: bool = Field(description="Whether the invoice has already been paid, based on any payment status indicators in the text")

    # before validating the field, call this function
    @field_validator("invoice_date", "due_date", mode="before")
    @classmethod
    def normalize_date(cls, v):
        if isinstance(v, str):
            return date_parser.parse(v)
        return v

In [ ]:
extracted_invoice: Invoice = instructor_client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    response_model=Invoice,
    messages=[
        {"role": "user", "content": f"Extract the invoice details from this text:\n\n{text}"}
    ],
)

print(extracted_invoice)
print("\nTotal amount:", extracted_invoice.total_amount)
print("Is paid:", extracted_invoice.is_paid)
for item in extracted_invoice.line_items:
    print(f"- {item.item_name} ({item.service_type.value}): {item.quantity} x ${item.price}")

In [ ]:
print(extracted_invoice.model_dump_json(indent=2))

---
# 🎓 Wrap-up

You've now hands-on practiced:

1. **Loading & prompting** open LLMs locally on Colab, for summarization, QA, and classification
2. **Constraining generation** at the practical way (Pydantic + Instructor)
